فقط ستون هایی که لازم داریم را داخل فایل اکسل می اوریم 


In [1]:
from datasets import load_dataset
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

c:\Users\lenovo\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", None)

In [3]:
df = pd.read_feather("../Outputs/28_df.feather")

In [4]:
is_residential = df['cat2_slug'].isin(
    ['residential-rent', 'residential-sell']
)
is_commercial = df['cat2_slug'].isin(['commercial-rent', 'commercial-sell'])

conditions = [is_residential, is_commercial]
choices = ['مسکونی', 'تجاری']

df['نوع ملک'] = np.select(conditions, choices, default='سایر')

In [5]:

try:
    cities_ref = pd.read_excel("divar_city_mapping_fa_en.xlsx")

    df['city_clean'] = df['city_slug'].astype(str).str.lower().str.strip()
    cities_ref['slug_clean'] = (
        cities_ref['city_slug'].astype(str).str.lower().str.strip()
    )

    cities_ref = cities_ref.drop_duplicates(subset=['slug_clean'])

    df = df.merge(
        cities_ref[['slug_clean', 'city_name_fa']],
        left_on='city_clean',
        right_on='slug_clean',
        how='left',
    )

    df = df.rename(columns={'city_name_fa': 'city_fa'}).drop(
        columns=['city_clean', 'slug_clean']
    )

    df['city_fa'] = df['city_fa'].fillna(df['city_slug'])

    print('ترجمه شهرها با موفقیت انجام شد.')

except Exception as e:
    print('خطا در دریافت یا پردازش فایل مرجع:', e)

ترجمه شهرها با موفقیت انجام شد.


In [6]:
df.columns

Index(['cat2_slug', 'cat3_slug', 'city_slug', 'neighborhood_slug',
       'created_at_month', 'user_type', 'description', 'title', 'rent_mode',
       'rent_value',
       ...
       'amenity_level', 'luxury_amenity_count', 'luxury_amenity_level',
       'city_neighborhood', 'full_property_type', 'month_number', 'month_name',
       'season', 'نوع ملک', 'city_fa'],
      dtype='object', length=102)

has_discount

In [ ]:
feature_columns = [
    "cat2_slug",
    "cat3_slug",
    "city_slug",
    "city_fa",
    "neighborhood_slug",
    "user_type",
    "property_type",
    'نوع ملک',
    'building_size',
    'land_size',
    "rent_credit_transform",
    "price_regime",
    "price_status",
    "sale_price",
    "sale_price_per_sqm",
    "equivalent_monthly_rent",
    "equivalent_deposit",
    "has_newly_built",
    "has_never_lived",
    "has_urgent",
    "has_swap",
    "has_below_market",
    "building_age",
    "created_at_month",
    "age_group",
    "area_group",
    "amenity_count",
    "amenity_level",
    "luxury_amenity_count",
    "luxury_amenity_level",
    "city_neighborhood",
    "full_property_type",
    "month_number",
    "month_name",
    "season",
    "is_probable_duplicate_to_remove",
    "is_exact_duplicate",
    "is_rebuilt",
    'sale_price_per_sqm_outlier',
       'equivalent_monthly_rent_outlier', 'equivalent_deposit_outlier',
       'building_size_outlier', 'outlier_price_flag', 'outlier_area_flag','coordinates_outlier',
       'location_latitude', 'location_longitude', 'location_radius',
]

In [14]:
features_df = df[feature_columns].copy()

In [15]:
features_df.to_csv("../Outputs/final2_df.csv", index=True, encoding="utf-8-sig")

In [10]:
features_df.head(10000).to_csv("../Outputs/final_10000_df.csv", index=True, encoding="utf-8-sig")